# Mount Google Drive (only for checkpoints, dataset comes from GitHub)
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the repository (includes dataset in data/hit-uav/)
import os
if os.path.exists('SGGF-Net'):
    print('Repository already exists, pulling latest changes...')
    %cd SGGF-Net
    !git pull origin main
else:
    !git clone https://github.com/HarishSankarK/SGGF-Net.git
    %cd SGGF-Net

# Verify dataset is included
!ls -la data/hit-uav/ 2>/dev/null && echo "✓ Dataset found in repository!" || echo "⚠ Dataset not found"


In [ ]:
# Verify we're in the right directory
import os
print(f"Current directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")


In [ ]:
# Setup TPU Runtime and Install PyTorch XLA
# Make sure TPU runtime is enabled: Runtime → Change runtime type → TPU

import os
print("Checking TPU runtime...")
print(f"COLAB_TPU_ADDR: {os.environ.get('COLAB_TPU_ADDR', 'Not set (TPU runtime not enabled)')}")

# Install PyTorch XLA for TPU support
print("\nInstalling PyTorch XLA for TPU...")
!pip install torch torchvision
!pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

# Install other required dependencies
print("\nInstalling other dependencies...")
!pip install numpy pillow opencv-python tqdm matplotlib scipy

# Verify TPU setup
# IMPORTANT: We check TPU availability but don't initialize it here
# This prevents "Device or resource busy" errors when train_tpu.py runs
is_tpu_available = False
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print('\n✓ torch_xla imported successfully')
    
    # Check environment variable first (more reliable in Colab)
    tpu_addr = os.environ.get('COLAB_TPU_ADDR', '')
    if tpu_addr:
        print(f'✓ TPU runtime environment detected: {tpu_addr}')
        is_tpu_available = True
    else:
        # Try to detect TPU by checking if we can access it (without initializing)
        # Note: Some Colab TPU setups work even without COLAB_TPU_ADDR
        try:
            # Try to get device info without full initialization
            # This is a lightweight check
            try:
                # Check if TPU devices are available in the system
                import subprocess
                result = subprocess.run(['test', '-e', '/dev/vfio/0'], capture_output=True)
                if result.returncode == 0:
                    print('✓ TPU device detected in system')
                    is_tpu_available = True
                else:
                    print('⚠ TPU device not detected in system')
            except:
                # If subprocess fails, try minimal torch_xla check
                # But don't call device() as it initializes
                pass
        except:
            pass
    
    # Only try to get device info if we detected TPU availability
    if is_tpu_available:
        print('✓ TPU is available')
        print('  Note: TPU will be initialized by train_tpu.py to avoid conflicts')
        print('  (Initializing here can cause "Device or resource busy" errors)')
        print('✓ TPU cores: Will be detected during training')
        print('  (TPU v5e-1 has 8 cores, but detection happens in train_tpu.py)')
    else:
        print('\n⚠ TPU runtime not detected')
        print('  If you want to use TPU:')
        print('  1. Runtime → Change runtime type → Select "TPU" → Save')
        print('  2. Runtime → Restart runtime')
        print('  3. Re-run this cell')
        
except ImportError:
    print('\n⚠ torch_xla not found - installation may have failed')
    is_tpu_available = False
except Exception as e:
    print(f'\n⚠ Error checking TPU: {e}')
    print('  This might be normal if TPU runtime is not enabled')
    is_tpu_available = False

if is_tpu_available:
    print('\n✅ TPU setup complete! You can now use train_tpu.py')
    print('   The script will properly initialize TPU when training starts')
else:
    print('\n⚠ TPU not detected - train_tpu.py will fall back to GPU/CPU')
    print('   Training will still work, just slower than TPU')


In [ ]:
# Dataset is already in the repository!
# Just verify it's there
import os

if os.path.exists('data/hit-uav'):
    print("✓ Dataset found in data/hit-uav/")
    try:
        train_img_dir = 'data/hit-uav/images/train'
        train_label_dir = 'data/hit-uav/labels/train'
        if os.path.exists(train_img_dir):
            num_images = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"  Images: {num_images} training images")
        else:
            print(f"  ⚠ Images directory not found: {train_img_dir}")
            
        if os.path.exists(train_label_dir):
            num_labels = len([f for f in os.listdir(train_label_dir) if f.endswith('.txt')])
            print(f"  Labels: {num_labels} training labels")
        else:
            print(f"  ⚠ Labels directory not found: {train_label_dir}")
            
        print("\nDataset structure:")
        !ls -la data/hit-uav/
    except Exception as e:
        print(f"⚠ Error checking dataset: {e}")
        print("  But dataset directory exists, continuing...")
else:
    print("⚠ Dataset not found. Make sure you've pushed it to GitHub.")


## Step 5: Start Training


In [ ]:
## Step 5: Start Training

# Setup checkpoint directory in Google Drive
import os
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(drive_checkpoint_dir, exist_ok=True)
print(f'Checkpoints will be saved to: {drive_checkpoint_dir}')

# Train the model on TPU v5e-1 (checkpoints saved to Drive)
# TPU v5e-1 OPTIMIZED: All optimizations applied for TPU v5e-1
# - Learning rate: 0.0005 (prevents NaN)
# - Batch size: 16 (TPU v5e-1 can handle batch_size=16-32 efficiently)
# - Max size: 800 (balanced for TPU memory)
# - Warmup + Cosine LR: faster convergence
# - GFEM optimized: patch_size=32, embed_dim=192, num_heads=6
# - TPU-specific: Uses torch_xla, parallel data loading, TPU mark_step
# TPU v5e-1 has 8 cores, 16GB HBM per core, and can train 3-5x faster than GPU!
# 
# NOTE: If TPU initialization fails, the script will automatically fall back to GPU/CPU
# 
# IMPORTANT: Make sure TPU runtime is enabled BEFORE running this cell!
# Steps:
# 1. Runtime → Change runtime type → Select "TPU" → Save
# 2. Runtime → Restart runtime
# 3. Re-run Cell 2 (TPU setup) and then this cell
# 
# Checking TPU availability...
import os

# Check environment variable
tpu_addr = os.environ.get('COLAB_TPU_ADDR', '')
if tpu_addr:
    print(f'✓ TPU runtime environment detected: {tpu_addr}')
else:
    print('⚠ TPU runtime environment NOT detected!')
    print('')
    print('📋 TO ENABLE TPU:')
    print('   1. Go to: Runtime → Change runtime type')
    print('   2. Select: TPU (not GPU or CPU)')
    print('   3. Click: Save')
    print('   4. Runtime → Restart runtime')
    print('   5. Re-run Cell 2 (TPU setup), then this cell')
    print('')
    print('⚠ The script will fall back to GPU/CPU if TPU is not available')

# Also check if torch_xla is installed (for better detection)
try:
    import torch_xla
    print('✓ torch_xla is installed')
    # Note: We don't initialize TPU here - let train_tpu.py handle it
    # This prevents "Device or resource busy" errors
except ImportError:
    print('⚠ torch_xla not found - did you run Cell 2?')
    print('   Run Cell 2 first to install torch_xla dependencies')

!python scripts/train_tpu.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --num_classes 6 \
    --batch_size 16 \
    --grad_accum_steps 1 \
    --num_epochs 50 \
    --lr 0.0005 \
    --max_size 800 \
    --warmup_epochs 5 \
    --checkpoint_dir {drive_checkpoint_dir} \
    --val_freq 10 \
    --use_amp


## Step 6: Evaluate Model


### Option B: TPU Training Command

**Requirements:**
- TPU runtime enabled (Runtime → Change runtime type → TPU)
- Step 2B completed successfully
- TPU initialization successful


In [ ]:
# Train the model on TPU v5e-1 (checkpoints saved to Drive)
# TPU v5e-1 OPTIMIZED: All optimizations applied for TPU v5e-1
# - Learning rate: 0.0005 (prevents NaN)
# - Batch size: 16 (TPU v5e-1 can handle batch_size=16-32 efficiently)
# - Max size: 800 (balanced for TPU memory)
# - Warmup + Cosine LR: faster convergence
# - GFEM optimized: patch_size=32, embed_dim=192, num_heads=6
# - TPU-specific: Uses torch_xla, parallel data loading, TPU mark_step
# TPU v5e-1 has 8 cores, 16GB HBM per core, and can train 3-5x faster than GPU!
# 
# NOTE: If TPU initialization fails, the script will automatically fall back to GPU/CPU
!python scripts/train_tpu.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --num_classes 6 \
    --batch_size 16 \
    --grad_accum_steps 1 \
    --num_epochs 50 \
    --lr 0.0005 \
    --max_size 800 \
    --warmup_epochs 5 \
    --checkpoint_dir {drive_checkpoint_dir} \
    --val_freq 10 \
    --use_amp


In [ ]:
# Evaluate on test set (using checkpoint from Drive)
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

!python scripts/evaluate.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --checkpoint {drive_checkpoint_dir}/best.pth \
    --num_classes 6 \
    --split test


## Step 7: Resume Training from Drive Checkpoint


In [ ]:
# Resume training from a checkpoint saved in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

# Check if checkpoint exists
import os
latest_checkpoint = f'{drive_checkpoint_dir}/latest.pth'
best_checkpoint = f'{drive_checkpoint_dir}/best.pth'

if os.path.exists(latest_checkpoint):
    resume_from = latest_checkpoint
    print(f'Resuming from: {resume_from}')
elif os.path.exists(best_checkpoint):
    resume_from = best_checkpoint
    print(f'Resuming from: {resume_from}')
else:
    resume_from = None
    print('No checkpoint found, starting fresh training')

# Resume training on GPU (using GPU-optimized settings)
# All optimizations applied: LR=0.0005, max_size=800, patch_size=32, batch_size=1 for GPU
if resume_from:
    !python scripts/train.py \
        --dataset hituav \
        --data_dir data/hit-uav \
        --num_classes 6 \
        --batch_size 1 \
        --grad_accum_steps 1 \
        --num_epochs 50 \
        --lr 0.0005 \
        --max_size 800 \
        --warmup_epochs 5 \
        --checkpoint_dir {drive_checkpoint_dir} \
        --resume {resume_from} \
        --device cuda \
        --val_freq 10 \
        --use_amp
else:
    print('No checkpoint to resume from. Run Step 5 to start training.')


## Checkpoint Management

Checkpoints are automatically saved to Google Drive at:
`/content/drive/MyDrive/SGGF-Net-checkpoints/`

- `latest.pth` - Latest checkpoint (every epoch)
- `best.pth` - Best model based on mAP

These persist even after Colab session ends!


### Option B: Resume TPU Training


In [ ]:
# Resume training on TPU v5e-1 (using TPU-optimized settings)
# All optimizations applied: LR=0.0005, max_size=800, patch_size=32, batch_size=16 for TPU v5e-1
if resume_from:
    !python scripts/train_tpu.py \
        --dataset hituav \
        --data_dir data/hit-uav \
        --num_classes 6 \
        --batch_size 16 \
        --grad_accum_steps 1 \
        --num_epochs 50 \
        --lr 0.0005 \
        --max_size 800 \
        --warmup_epochs 5 \
        --checkpoint_dir {drive_checkpoint_dir} \
        --resume {resume_from} \
        --val_freq 10 \
        --use_amp
else:
    print('No checkpoint to resume from. Run Step 5 to start training.')


In [ ]:
# List checkpoints in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
import os

if os.path.exists(drive_checkpoint_dir):
    print(f"Checkpoints in Drive ({drive_checkpoint_dir}):")
    checkpoints = os.listdir(drive_checkpoint_dir)
    for ckpt in checkpoints:
        if ckpt.endswith('.pth'):
            size = os.path.getsize(f'{drive_checkpoint_dir}/{ckpt}') / (1024*1024)  # MB
            print(f"  - {ckpt} ({size:.2f} MB)")
else:
    print(f"Checkpoint directory not found: {drive_checkpoint_dir}")
    print("Run Step 5 to start training and create checkpoints.")
